In [2]:
import geopandas as gpd
import numpy as np
import pandas as pd
import geojson_validator
from shapely.ops import unary_union
import pandas as pd
import numpy as np
from shapely.geometry import Point, MultiPolygon
import geopandas as gpd
from geopandas import GeoDataFrame
from fuzzywuzzy import process

In [3]:
tehsil_gdf = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\HP_losses_and_damages\fixed_RHR_dupVertices_4326_reduced.geojson")
losses_df = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\HP_losses_and_damages\HP_losses_and_damages.csv')

In [5]:
district_gdf = tehsil_gdf.dissolve(by='District', as_index=False)


In [6]:
# Function to fix invalid geometries
def fix_geometry(geom):
    if geom is None:
        return None
    if not geom.is_valid:
        # Attempt to fix using buffer(0)
        geom = geom.buffer(0)
    return geom if geom.is_valid else None

# Check and fix geometries
def clean_geometries(gdf):
    # Check for missing or invalid geometries
    gdf['geometry_fixed'] = gdf['geometry'].apply(fix_geometry)

    # Drop rows with irreparable (None) geometries
    gdf = gdf.dropna(subset=['geometry_fixed'])

    return gdf

def geometry_to_text(geom):
    if geom is None:
        return None
    
    # Handle Polygon and LineString geometries
    if geom.geom_type == 'Polygon':
        coords = list(geom.exterior.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    elif geom.geom_type == 'LineString':
        coords = list(geom.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    # Handle MultiPolygon and MultiLineString geometries
    elif geom.geom_type in ['MultiPolygon', 'MultiLineString']:
        all_coords = []
        for part in geom.geoms:  # Loop through each sub-geometry
            if part.geom_type == 'Polygon':
                coords = list(part.exterior.coords)
            else:
                coords = list(part.coords)
            all_coords.append([[lon, lat] for lon, lat in coords])
        return str(all_coords)
    
    # Catch other geometry types if necessary
    else:
        return None

def get_best_match(block_name, block_names):
    match, score = process.extractOne(block_name, block_names)
    return match if score > 80 else None  # Adjust threshold as needed

In [7]:
import json

# Flatten nested geometries if required
fixed_geo_flattened = {
    key: [json.dumps(item) if isinstance(item, dict) else item for item in value]
    for key, value in district_gdf.items()
}
geo_fixed = pd.DataFrame.from_dict(fixed_geo_flattened)

In [8]:
# Simplify MultiPolygon by selecting the largest Polygon
def simplify_multipolygon(geometry):
    if isinstance(geometry, MultiPolygon):
        # Select the largest Polygon by area
        return max(geometry.geoms, key=lambda geom: geom.area)
    return geometry

# Apply the simplification
district_gdf['geometry'] = district_gdf['geometry'].apply(simplify_multipolygon)
tehsil_gdf['geometry'] = tehsil_gdf['geometry'].apply(simplify_multipolygon)

district_fixed = gpd.GeoDataFrame(district_gdf, geometry='geometry')
tehsil_fixed = gpd.GeoDataFrame(tehsil_gdf, geometry='geometry')


In [14]:
tehsil_fixed = tehsil_fixed.rename(columns={'geometry': 'tehsil_geometry'})
district_fixed = district_fixed.rename(columns={'geometry': 'district_geometry'})

In [15]:
merged_gdf = losses_df.merge(district_fixed[['district_geometry','District']], right_on=['District'], left_on=['DISTRICT_FINALISED'], how='left')
merged_gdf_2 = merged_gdf.merge(tehsil_fixed[['tehsil_geometry','TEHSIL']], right_on=['TEHSIL'], left_on=['TEHSIL_FINALISED'], how='left')
#merged_gdf = merged_gdf.drop(columns=['dtname','tender_revenueci_location'])
merged_gdf_2

,disaster_id,department_id,district_id,status,croppedarea,crop_lossless33,cropsownarea,crop_33_above,kharif_id,totalAreadamaged,...,loss_type,countno,add_RelifAmount,structure_type,DISTRICT_FINALISED,TEHSIL_FINALISED,district_geometry,District,tehsil_geometry,TEHSIL
0,4682.0,3.0,23.0,Active,0.0,0.0,3985.0,0.0,1.0,0.0,...,NaN,NaN,NaN,NaN,SHIMLA,NaN,None,NaN,None,NaN
1,4682.0,3.0,23.0,Active,0.0,0.0,0.0,0.0,1.0,0.0,...,NaN,NaN,NaN,NaN,SHIMLA,NaN,None,NaN,None,NaN
2,4682.0,3.0,23.0,Active,0.0,0.0,0.0,0.0,1.0,0.0,...,NaN,NaN,NaN,NaN,SHIMLA,NaN,None,NaN,None,NaN
3,4682.0,3.0,23.0,Active,0.0,0.0,310.0,0.0,1.0,0.0,...,NaN,NaN,NaN,NaN,SHIMLA,NaN,None,NaN,None,NaN
4,4682.0,3.0,23.0,Active,0.0,0.0,200.0,0.0,1.0,0.0,...,NaN,NaN,NaN,NaN,SHIMLA,NaN,None,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3303,26666.0,40.0,20.0,Active,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,5694.0,1.0,Kachha House,KULLU,KULLU,None,NaN,None,NaN
3304,26841.0,40.0,20.0,Active,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,5692.0,1.0,Kachha House,KULLU,KULLU,None,NaN,None,NaN
3305,26841.0,40.0,20.0,Active,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,5662.0,1.0,Kachha House,KULLU,KULLU,None,NaN,None,NaN
3306,26841.0,40.0,20.0,Active,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,5689.0,1.0,Kachha House,KULLU,KULLU,None,NaN,None,NaN


In [17]:
merged_gdf_2['district_polygons'] = merged_gdf_2['district_geometry'].apply(geometry_to_text)
merged_gdf_2['tehsil_polygons'] = merged_gdf_2['tehsil_geometry'].apply(geometry_to_text)
cleaned_gdf = merged_gdf_2.drop(columns=['district_geometry','tehsil_geometry'])

In [9]:
merged_gdf = merged_gdf.dropna(subset =['object_id'])

KeyError: ['object_id']

In [27]:
from datetime import date, timedelta, datetime


#snapshot_ld_dist['datetime'] = pd.to_datetime(snapshot_ld_dist['timeperiod'], format='%Y_%m')
merged_gdf['timeperiod'] = merged_gdf['month'].str.replace('_', '-') #+ '-01'

# Step 2: Convert the modified column to datetime format
#snapshot_ld_dist['timeperiod_iso'] = pd.to_datetime(snapshot_ld_dist['timeperiod_iso'], format='%Y-%m-%d')
merged_gdf['timeperiod'] = pd.to_datetime(merged_gdf['timeperiod'], format='%Y-%m')


merged_gdf['timeperiod'] = merged_gdf['timeperiod'].dt.strftime('%Y-%m')

In [28]:
merged_gdf.columns = merged_gdf.columns.str.lower().str.replace(' ', '_')
merged_gdf = merged_gdf.rename(columns={'contract_date_:':'contract_date','bid_validity(days)':'bid_validity_days','tender_value_in_₹':'tender_value_in_rupees'})
merged_gdf['tender_value_in_rupees'] = merged_gdf['tender_value_in_rupees'].str.replace(',', '')
merged_gdf['tender_value_in_rupees'] = merged_gdf['tender_value_in_rupees'].astype(float)
merged_gdf = merged_gdf.dropna(subset=['awarded_value', 'tender_value_in_rupees','district_finalised'])
merged_gdf['awarded_value'] = merged_gdf['awarded_value'].str.replace(',', '')
merged_gdf['awarded_value'] = merged_gdf['awarded_value'].astype(float)
merged_gdf

,unnamed:_0,tender_id,tender_externalreference,tender_title,work_description,tender_category,tender_type,form_of_contract,product_category,is_multi_currency_allowed_for_boq,...,district_finalised,tender_villages,tender_block,tender_subdistrict,tender_revenueci,hq_flag,revenue_circle_finalised,geometry,polygons,timeperiod
0,159,2017_DoWR_2083_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal 1,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
1,160,2017_DoWR_2238_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal Drainage Basin Ph-II,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
2,210,2018_DoWR_5796_1,HAILAKANDI/SDRF/2017-18/1,IM at Kalinagar Pk-1,Immediate measures to dyke along l/b of river ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,"'MOHANPUR', 'KALINAGAR'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
3,372,2018_DoWR_6310_2,HAILAKANDI/RIDF-XXIII/1,A E Measures to Protect Sahabad-Rongpur area,Anti Erosion Measures to Protect Sahabad-Rongp...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,"'SAHABAD', 'RONGPUR', 'Rongpur'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2019-09
4,552,2019_DoWR_11496_1,HAILAKANDI/2018-19/SDRF/II,IM at Matijuri,Immediate measures to Restoration for damages ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,"'MATIJURI', 'Matijuri'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2019-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2778,2673,2023_SCD_34601_8,Tender/NIT/Pt/2023-24/7736,EARTHEN EMBANKMENT/GUIDE BUND AT MORAN CHUTIA ...,EARTHEN EMBANKMENT/GUIDE BUND AT MORAN CHUTIA ...,Works,Open Tender,Item Rate,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-08
2779,2680,2024_DoWR_36023_2,GOLAGHAT/2023-24/NIDA/I,Anti erosion measures to protect Amguri Basapa...,Anti erosion measures to protect Amguri Basapa...,Works,Open Tender,Works,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-08
2780,2682,2024_DoWR_36478_1,DHEMAJI/2023-24/NIDA/II,Extension of Gainadi L/B embankment from Sumon...,Extension of Gainadi L/B embankment from Sumon...,Works,Open Tender,Works,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-08
2781,2775,2024_ICD_38053_1,505342/173 dated 09.07.2024,"Construction of Boundary Wall, Land Developmen...","Construction of Boundary Wall, Land Developmen...",Works,Open Tender,Item Rate,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-12


In [18]:
cleaned_gdf.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\HP_losses_and_damages\hp_losses_polygons.csv', index=False)